# Mouse exon and intron smiFISH probe sets

Designs exon and intron sets for Spen, Arid5b, Jarid2, Sfmbt2, Sfmbt1 and Mdm4, plus an exon-only Malat1 control. The mm10 RefGene Malat1 transcript has no intron. The integrated Python Oligostan port generates all filter-eligible candidates across annotated regions, and every candidate is BLAST checked. The minimum-span add-on then selects the narrowest genomic window of 30 BLAST-verified probes per available set. No 40-probe preselection is applied. Coordinates are mm10/GRCm38, 1-based inclusive.

A selected probe must have an exact full-length BLAST hit at its intended locus and no other alignment covering at least 80% of its length at at least 90% identity. Review the hit audit and order sheets before synthesis. The full mouse Spen comparison to the original R script passed under matched fixed −32, masking-off settings; the minimum-span selector is a Python add-on, not part of that parity claim.

Run top to bottom. Create an ignored `.smifish-local.json` beside `oligostan/` using `oligostan/mouse_smifish.example.json`. Point it to the mm10 FASTA, `.fai`, GTF, BLAST database, `blastn`, and output directory. Use a kernel with Biopython, pandas, Rich and Matplotlib installed. Rich progress bars appear during annotation loading, candidate generation, batched BLAST, export, and report drawing.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display, FileLink
import matplotlib  # Fail early if the report dependency is missing.

workdir = Path.cwd()
package_parent = next((p for p in (workdir, workdir / "probe_and_primer_design", workdir.parent / "probe_and_primer_design") if (p / "oligostan" / "__init__.py").is_file()), None)
if package_parent is None:
    raise FileNotFoundError("Run this notebook from the repository or probe_and_primer_design directory")
sys.path.insert(0, str(package_parent))

from oligostan.mouse_smifish import (
    GENES, load_run_config, load_models, generate_candidates,
    adaptive_blast_verify, write_outputs,
)

from oligostan.minimum_span import select_minimum_span_sets

config = load_run_config()
print("Reference: mm10/GRCm38; targets:", ", ".join(GENES))
print("Output parent:", config["output_parent"])


## 1. Load transcript annotations

The longest spliced RefGene transcript is chosen for each gene. Exon probes remain within a single exon. Intronic regions are stripped of any exon from another annotated isoform.


In [ ]:
models = load_models(config["gtf"])
display(__import__("pandas").DataFrame([
    {"gene": gene, "transcript": model["transcript_id"], "chrom": model["chrom"],
     "strand": model["strand"], "exons": len(model["exon"]),
     "eligible_introns": len(model["intron"])}
    for gene, model in models.items()
]))


## 2. Design the complete Oligostan candidate pool

The core algorithm designs 26–32 nt probes near dG37 = −32. The package's GC 0.4–0.6 and PNAS 1/2/4 settings are the baseline; other candidates receive documented relaxation tiers. All eligible exon and intron tiles are processed without a candidate-count stop, so the later span search sees the full generated pool.


In [ ]:
candidates = generate_candidates(models, fasta=config["fasta"])
display(candidates.groupby(["gene", "region"]).size().rename("filtered_candidates").reset_index())


## 3. BLAST-verify every filtered candidate

`blastn-short` searches both strands of mm10 with dust and soft masking disabled. All quality tiers are searched in batches before any window selection. A probe passes only with an exact full-length intended hit and no strong off-target. The complete hit audit is exported.


In [ ]:
verified, blast_hits, blast_command = adaptive_blast_verify(candidates, blastn=config["blastn"], database=config["blast_db"], threads=8)
display(verified.groupby(["gene", "region"])["blast_verified"].agg(["sum", "count"]).reset_index())


## 4. Select the smallest 30-probe genomic window

The add-on module compares every consecutive 30-probe window in each coordinate-sorted BLAST-passing pool. It minimizes the inclusive genomic span and uses quality tier and Oligostan score only to break equal-span ties. A set with fewer than 30 passing probes is flagged and never padded with unverified probes.


In [ ]:
selected, summary = select_minimum_span_sets(verified, models, probes_per_set=30)
display(summary)
if not summary["meets_minimum"].all():
    print("Some sets have fewer than 30 BLAST-verified probes; inspect set_summary.csv before ordering.")
else:
    print(f"All {len(summary)} available sets have 30 BLAST-verified probes.")


## 5. Export probe sets and genomic coverage report

A fresh subfolder is created under the configured `output_parent`. The root includes all designed candidates, BLAST statuses and hits, the 30-probe selected sets, summaries, and manifest. Package functions also save `coverage_report.pdf`, `coverage_summary.csv`, and individual genomic-coordinate plots under `coverage_plots/`.


In [ ]:
output_dir = write_outputs(models, candidates, verified, blast_hits, selected, summary, blast_command, output_parent=config["output_parent"], reference_paths=config)
print("Saved:", output_dir)
display(summary[["gene", "region", "selected_count", "genomic_span_bp", "meets_minimum"]])
display(FileLink(output_dir / "coverage_report.pdf"))
